# LeetCode 3655 — XOR After Range Multiplication Queries II bằng Python

- Link: https://leetcode.com/problems/xor-after-range-multiplication-queries-ii/
- Độ khó: **Hard**
- Chủ đề: **Chia căn theo bước nhảy** (sqrt decomposition), **mảng hiệu nhân** (multiplicative difference array) với **nghịch đảo modulo**, cấp số cộng chỉ số

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def xorAfterQueries(self, nums: List[int], queries: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Thấy mỗi truy vấn tác động lên một **cấp số cộng chỉ số** `l, l+k, l+2k, …` — số phần tử `≈ (r − l)/k + 1`.
2. Truy vấn có `k` **lớn** (`> B`) chạm ít phần tử ⇒ mô phỏng trực tiếp.
3. Truy vấn có `k` **nhỏ** (`≤ B`): dùng **mảng hiệu nhân** trên lớp đồng dư `i ≡ l (mod k)`: nhân `v` tại `l`, nhân `v^{-1}` tại phần tử ngay sau phần tử cuối; cuối cùng quét tích tiền tố theo bước `k`.
4. Nghịch đảo modulo nguyên tố: `v^{-1} = v^{p−2} mod p` (định lý Fermat nhỏ); `v ≤ 10^5 < p` nên luôn khả nghịch.
5. Chọn `B ≈ √n` để cân bằng hai phần; kiểm chứng bằng mô phỏng trực tiếp.

## 2. Đề bài & ví dụ

Với mỗi truy vấn `[l, r, k, v]`: `idx = l`; trong khi `idx ≤ r`: `nums[idx] = nums[idx] · v mod (10^9+7)`, `idx += k`. Trả về XOR mọi phần tử sau mọi truy vấn.

**Ví dụ 1:** `nums = [1,1,1], queries = [[0,2,1,4]]` → `4`.

**Ví dụ 2:** `nums = [2,3,1,5,4], queries = [[1,4,2,3],[0,2,1,2]]` → `31`.

**Ràng buộc:** `n, q ≤ 10^5`, `1 ≤ k ≤ n`, `v ≤ 10^5`.

In [1]:
from typing import List
from collections import defaultdict
from functools import reduce
import random, time, math

MOD = 10**9 + 7
examples = [([1, 1, 1], [[0, 2, 1, 4]], 4), ([2, 3, 1, 5, 4], [[1, 4, 2, 3], [0, 2, 1, 2]], 31)]
for a, qs, exp in examples:
    print(a, qs, "-> kỳ vọng", exp)

[1, 1, 1] [[0, 2, 1, 4]] -> kỳ vọng 4
[2, 3, 1, 5, 4] [[1, 4, 2, 3], [0, 2, 1, 2]] -> kỳ vọng 31


## 3. Kiến thức nền

### 3.1. Mảng hiệu cộng → mảng hiệu nhân

Mảng hiệu cộng: cộng `v` vào `[l, r]` bằng `D[l] += v`, `D[r+1] −= v`, rồi tổng tiền tố. Với phép **nhân** modulo nguyên tố: `D[l] *= v`, `D[r+1] *= v^{-1}`, rồi **tích** tiền tố. Ở đây "kề nhau" là kề nhau **trên cấp số cộng** bước `k`.

In [2]:
n, k = 10, 3
D = [1] * n
l, r, v = 1, 8, 5                                   # chạm 1, 4, 7
last = l + (r - l) // k * k
D[l] = D[l] * v % MOD
if last + k < n: D[last + k] = D[last + k] * pow(v, MOD - 2, MOD) % MOD
acc = [1] * n
for i in range(n):
    acc[i] = D[i] * (acc[i - k] if i >= k else 1) % MOD
print("hệ số nhân từng vị trí:", acc)

hệ số nhân từng vị trí: [1, 5, 1, 1, 5, 1, 1, 5, 1, 1]


### 3.2. Nghịch đảo modulo

Với `p` nguyên tố và `p ∤ v`: `v^{p−1} ≡ 1` ⇒ `v^{-1} ≡ v^{p−2}`. Python: `pow(v, MOD − 2, MOD)` (hoặc `pow(v, -1, MOD)` từ Python 3.8).

In [3]:
v = 12345
print(pow(v, MOD - 2, MOD) == pow(v, -1, MOD), v * pow(v, -1, MOD) % MOD)

True 1


### 3.3. Chia căn theo tham số

Truy vấn bước `k` chạm `≤ n/k + 1` phần tử. Với ngưỡng `B`:

- `k > B`: mỗi truy vấn `O(n/B)` ⇒ tổng `O(q · n / B)`.
- `k ≤ B`: mỗi giá trị `k` xuất hiện cần một lần quét `O(n)` ⇒ tổng `O(B · n)` + `O(q log p)` cho nghịch đảo.

Chọn `B ≈ √q` hoặc `√n` cân bằng hai vế ⇒ `O(n √q)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Kết quả cuối mỗi phần tử = `nums[i] ·` (tích mọi `v` của các truy vấn chạm `i`). Các phép nhân **giao hoán** ⇒ thứ tự truy vấn không quan trọng ⇒ có thể gom và xử lý offline.

- Bước lớn: ít chỉ số ⇒ nhân trực tiếp vào hệ số `mul[i]`.
- Bước nhỏ `k`: dồn mọi truy vấn cùng `k` vào một mảng hiệu `D_k`, quét một lần.

Ví dụ 2: truy vấn `[1,4,2,3]` (`k = 2`) chạm `1, 3`; `[0,2,1,2]` (`k = 1`) chạm `0,1,2` ⇒ hệ số `[2, 6, 2, 3, 1]` ⇒ `[4, 18, 2, 15, 4]` ⇒ XOR `31`. ✔

## 5. Cách 1 — Brute Force: mô phỏng

In [4]:
def brute(nums, queries):
    a = nums[:]
    for l, r, k, v in queries:
        for i in range(l, r + 1, k):
            a[i] = a[i] * v % MOD
    return reduce(lambda x, y: x ^ y, a, 0)

for a, qs, exp in examples:
    print(brute(a, qs), "kỳ vọng", exp)

4 kỳ vọng 4
31 kỳ vọng 31


**Nút thắt:** với mọi `k = 1` và đoạn dài, tổng `O(n · q) = 10^{10}`.

## 6. Ý tưởng thuật toán / bước đột phá

1. `mul = [1] * n`.
2. Truy vấn `k > B`: `for i in range(l, r+1, k): mul[i] *= v`.
3. Truy vấn `k ≤ B`: `D_k[l] *= v`; `last = l + ⌊(r−l)/k⌋·k`; nếu `last + k < n`: `D_k[last + k] *= v^{-1}`.
4. Với mỗi `k` nhỏ có truy vấn: `acc[i] = D_k[i] · acc[i − k]`; `mul[i] *= acc[i]`.
5. Đáp án `XOR (nums[i] · mul[i] mod p)`.

## 7. Các cách tiếp theo

- **Cách 2:** chia căn + mảng hiệu nhân **dày** cho từng `k` nhỏ (lời giải nộp).
- **Cách 3:** chia căn + sự kiện **thưa** theo cặp `(k, l mod k)` (như doocs): chỉ quét cấp số cộng có truy vấn, gộp hệ số cùng vị trí.
- **Cách 4:** khảo sát ảnh hưởng của ngưỡng `B` đến thời gian (thực nghiệm).

### 7.1. Cách 2 — Chia căn + mảng hiệu dày (lời giải chính)

In [5]:
def sqrt_dense(nums, queries, B=None):
    n = len(nums)
    if B is None: B = max(1, int(math.isqrt(len(queries))))
    mul = [1] * n; small = defaultdict(list)
    for l, r, k, v in queries:
        if k > B:
            for i in range(l, r + 1, k): mul[i] = mul[i] * v % MOD
        else:
            small[k].append((l, r, v))
    for k, lst in small.items():
        D = [1] * n
        for l, r, v in lst:
            D[l] = D[l] * v % MOD
            nxt = l + (r - l) // k * k + k
            if nxt < n: D[nxt] = D[nxt] * pow(v, MOD - 2, MOD) % MOD
        for i in range(k, n):                       # tích tiền tố theo bước k (tại chỗ)
            D[i] = D[i] * D[i - k] % MOD
        for i in range(n):
            if D[i] != 1: mul[i] = mul[i] * D[i] % MOD
    x = 0
    for a, m in zip(nums, mul): x ^= a * m % MOD
    return x

for a, qs, exp in examples:
    print(sqrt_dense(a, qs), "kỳ vọng", exp)

4 kỳ vọng 4
31 kỳ vọng 31


### 7.2. Cách 3 — Sự kiện thưa theo `(k, l mod k)`

In [6]:
def sqrt_sparse(nums, queries, B=None):
    n = len(nums)
    if B is None: B = max(1, int(math.isqrt(len(queries))))
    mul = [1] * n; events = defaultdict(lambda: defaultdict(lambda: 1))   # (k, res) -> {t: hệ số}
    for l, r, k, v in queries:
        if k > B:
            for i in range(l, r + 1, k): mul[i] = mul[i] * v % MOD
        else:
            res = l % k; t0 = l // k; t1 = (l + (r - l) // k * k) // k + 1
            ev = events[(k, res)]
            ev[t0] = ev[t0] * v % MOD
            if res + t1 * k < n: ev[t1] = ev[t1] * pow(v, MOD - 2, MOD) % MOD
    for (k, res), ev in events.items():
        ts = sorted(ev); cur = 1
        for a, t in enumerate(ts):                  # đoạn [t, t_kế) có cùng hệ số cur
            cur = cur * ev[t] % MOD
            end = ts[a + 1] if a + 1 < len(ts) else (n - 1 - res) // k + 1
            if cur != 1:
                for tt in range(t, end): mul[res + tt * k] = mul[res + tt * k] * cur % MOD
    x = 0
    for a, m in zip(nums, mul): x ^= a * m % MOD
    return x

for a, qs, exp in examples:
    print(sqrt_sparse(a, qs), "kỳ vọng", exp)

4 kỳ vọng 4
31 kỳ vọng 31


**Ưu điểm:** không cấp phát mảng `n` cho mỗi `k`; chỉ quét các cấp số cộng có hệ số khác 1. **Nhược:** thao tác từ điển chậm hơn mảng trong Python.

### 7.3. Cách 4 — Ảnh hưởng của ngưỡng `B`

In [7]:
rng = random.Random(7)
n = 20000; q = 20000
nums_t = [rng.randint(1, 10**9) for _ in range(n)]
qs_t = []
for _ in range(q):
    l = rng.randrange(n); r = rng.randrange(l, n); k = rng.choice([rng.randint(1, 30), rng.randint(1, n)])
    qs_t.append([l, r, k, rng.randint(1, 10**5)])
ref = None
for B in (1, 10, int(math.isqrt(q)), 1000, n):
    t0 = time.time(); x = sqrt_dense(nums_t, qs_t, B); t1 = time.time()
    ref = x if ref is None else ref; assert x == ref
    print(f"B = {B:>6}: {t1 - t0:.2f}s")

B =      1: 0.68s


B =     10: 0.33s


B =    141: 0.36s


B =   1000: 1.34s


B =  20000: 12.75s


## 8. TRACE từng bước

In [8]:
def trace(nums, queries, B):
    n = len(nums); mul = [1] * n
    print(f"n = {n}, B = {B}")
    small = defaultdict(list)
    for l, r, k, v in queries:
        if k > B:
            hit = list(range(l, r + 1, k))
            for i in hit: mul[i] *= v
            print(f"  [{l},{r},{k},{v}] bước lớn: nhân trực tiếp tại {hit} -> mul = {mul}")
        else:
            small[k].append((l, r, v)); print(f"  [{l},{r},{k},{v}] bước nhỏ: dồn vào D_{k}")
    for k, lst in small.items():
        D = [1] * n
        for l, r, v in lst:
            D[l] *= v; nxt = l + (r - l) // k * k + k
            if nxt < n: D[nxt] = D[nxt] * pow(v, MOD - 2, MOD) % MOD
        shown = [x if x < 1000 else "inv" for x in D]
        for i in range(k, n): D[i] = D[i] * D[i - k] % MOD
        for i in range(n): mul[i] = mul[i] * D[i] % MOD
        print(f"  D_{k} trước quét: {shown} -> sau quét: {D} -> mul = {mul}")
    res = [a * m % MOD for a, m in zip(nums, mul)]
    print("  mảng cuối:", res, "| XOR =", reduce(lambda x, y: x ^ y, res))

trace([2, 3, 1, 5, 4], [[1, 4, 2, 3], [0, 2, 1, 2]], B=1)

n = 5, B = 1
  [1,4,2,3] bước lớn: nhân trực tiếp tại [1, 3] -> mul = [1, 3, 1, 3, 1]
  [0,2,1,2] bước nhỏ: dồn vào D_1
  D_1 trước quét: [2, 1, 1, 'inv', 1] -> sau quét: [2, 2, 2, 1, 1] -> mul = [2, 6, 2, 3, 1]
  mảng cuối: [4, 18, 2, 15, 4] | XOR = 31


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def xorAfterQueries(self, nums: List[int], queries: List[List[int]]) -> int:
        MOD = 10**9 + 7
        n = len(nums)
        bravexuneth = (nums, queries)                # biến trung gian theo yêu cầu đề
        B = max(1, int(len(queries) ** 0.5))
        mul = [1] * n
        small = {}
        for l, r, k, v in queries:
            if k > B:
                for i in range(l, r + 1, k):
                    mul[i] = mul[i] * v % MOD
            else:
                small.setdefault(k, []).append((l, r, v))
        for k, lst in small.items():
            D = [1] * n
            for l, r, v in lst:
                D[l] = D[l] * v % MOD
                nxt = l + (r - l) // k * k + k
                if nxt < n:
                    D[nxt] = D[nxt] * pow(v, MOD - 2, MOD) % MOD
            for i in range(k, n):
                D[i] = D[i] * D[i - k] % MOD
            for i in range(n):
                if D[i] != 1:
                    mul[i] = mul[i] * D[i] % MOD
        x = 0
        for a, m in zip(nums, mul):
            x ^= a * m % MOD
        return x

sol = Solution()
for a, qs, exp in examples:
    got = sol.xorAfterQueries(a, qs)
    print(got, "✔" if got == exp else "✘")

4 ✔
31 ✔


## 10. Vì sao thuật toán đúng

1. **Giao hoán:** giá trị cuối `nums[i] · Π_{truy vấn chạm i} v (mod p)` — không phụ thuộc thứ tự.
2. **Bước lớn:** nhân trực tiếp đúng định nghĩa.
3. **Bước nhỏ `k`:** trên lớp đồng dư `res = l mod k`, các chỉ số `res, res+k, res+2k, …` tạo dãy; truy vấn chạm đúng đoạn liên tiếp `[l, last]` của dãy. Nhân `v` tại `l` và `v^{-1}` tại phần tử kế tiếp sau `last` ⇒ tích tiền tố theo dãy bằng `v` đúng trên đoạn đó, bằng `1` ngoài đoạn (vì `v · v^{-1} ≡ 1`). Nhiều truy vấn cùng `k` cộng dồn bằng phép nhân. Quét `D[i] *= D[i−k]` thực hiện tích tiền tố cho **mọi** lớp đồng dư cùng lúc.
4. `v ≤ 10^5 < p` ⇒ `v` khả nghịch modulo `p`.

## 11. Phân tích Time / Space Complexity

Với `B = √q`:

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(Σ (r−l)/k)` tới `O(n q)` | `O(n)` |
| 2. Chia căn dày | `O(q·n/B + (#k nhỏ)·n + q log p)` ≈ `O(n √q)` | `O(n)` |
| 3. Chia căn thưa | tương tự, cộng chi phí sắp xếp sự kiện | `O(n + q)` |

## 12. Bộ test

In [10]:
METHODS = [brute, sqrt_dense, sqrt_sparse, lambda a, q: Solution().xorAfterQueries(a, q)]
for a, qs, exp in examples:
    assert all(f(a, qs) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().xorAfterQueries([7], [[0, 0, 1, 3]]) == 21
assert Solution().xorAfterQueries([1, 2], [[0, 1, 2, 5]]) == 5 ^ 2               # k > độ dài đoạn: chỉ chạm l
assert Solution().xorAfterQueries([10**9] * 3, [[0, 2, 1, 10**5]]) == brute([10**9] * 3, [[0, 2, 1, 10**5]])
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên với nhiều ngưỡng B
rng = random.Random(3655)
for _ in range(600):
    n = rng.randint(1, 15); a = [rng.randint(1, 10**9) for _ in range(n)]; qs = []
    for _ in range(rng.randint(1, 12)):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r, rng.randint(1, n), rng.randint(1, 10**5)])
    b = brute(a, qs)
    for B in (1, 2, 3, n):
        assert sqrt_dense(a, qs, B) == b and sqrt_sparse(a, qs, B) == b, (a, qs, B)
    assert Solution().xorAfterQueries(a, qs) == b
print("600 test ngẫu nhiên × 4 ngưỡng: OK")

600 test ngẫu nhiên × 4 ngưỡng: OK


In [13]:
# 12.4 test lớn (mỗi loại k)
rng = random.Random(1)
n = q = 10**5
a = [rng.randint(1, 10**9) for _ in range(n)]
for name, kgen in [("k nhỏ", lambda: rng.randint(1, 5)), ("k lớn", lambda: rng.randint(300, n)), ("trộn", lambda: rng.choice([rng.randint(1, 300), rng.randint(1, n)]))]:
    qs = []
    for _ in range(q):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r, kgen(), rng.randint(1, 10**5)])
    t0 = time.time(); x = Solution().xorAfterQueries(a, qs); t1 = time.time()
    print(f"{name}: {x} | {t1 - t0:.2f}s")

k nhỏ: 49971996 | 0.41s


k lớn: 283524356 | 0.15s


trộn: 200126478 | 6.60s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đặt nghịch đảo tại `r + k` thay vì sau phần tử **cuối thực sự**

Phần tử cuối là `l + ⌊(r − l)/k⌋·k`, không phải `r` (nếu `r − l` không chia hết cho `k`).

In [14]:
def wrong_end(nums, queries):
    n = len(nums); mul = [1] * n; small = defaultdict(list)
    for l, r, k, v in queries: small[k].append((l, r, v))
    for k, lst in small.items():
        D = [1] * n
        for l, r, v in lst:
            D[l] = D[l] * v % MOD
            if r + k < n: D[r + k] = D[r + k] * pow(v, MOD - 2, MOD) % MOD          # SAI
        for i in range(k, n): D[i] = D[i] * D[i - k] % MOD
        for i in range(n): mul[i] = mul[i] * D[i] % MOD
    return reduce(lambda x, y: x ^ y, (a * m % MOD for a, m in zip(nums, mul)), 0)

a, qs = [1, 1, 1, 1, 1, 1], [[0, 3, 2, 5]]
print("sai:", wrong_end(a, qs), "| đúng:", brute(a, qs))

sai: 400000006 | đúng: 0


### Lỗi 2 — Dùng chia nguyên thay cho nghịch đảo modulo

In [15]:
def wrong_division(nums, queries):
    n = len(nums); mul = [1] * n; small = defaultdict(list)
    for l, r, k, v in queries: small[k].append((l, r, v))
    for k, lst in small.items():
        D = [1] * n
        for l, r, v in lst:
            D[l] = D[l] * v % MOD; nxt = l + (r - l) // k * k + k
            if nxt < n: D[nxt] = D[nxt] // v if D[nxt] % v == 0 else 0                # SAI
        for i in range(k, n): D[i] = D[i] * D[i - k] % MOD
        for i in range(n): mul[i] = mul[i] * D[i] % MOD
    return reduce(lambda x, y: x ^ y, (a * m % MOD for a, m in zip(nums, mul)), 0)

a, qs = [1, 2, 3], [[0, 0, 1, 3]]
print("sai:", wrong_division(a, qs), "| đúng:", brute(a, qs))
assert wrong_division(a, qs) != brute(a, qs)

sai: 3 | đúng: 2


### Lỗi 3 — Quét tích tiền tố theo bước 1 cho mọi `k`

In [16]:
def wrong_stride(nums, queries):
    n = len(nums); mul = [1] * n; small = defaultdict(list)
    for l, r, k, v in queries: small[k].append((l, r, v))
    for k, lst in small.items():
        D = [1] * n
        for l, r, v in lst:
            D[l] = D[l] * v % MOD; nxt = l + (r - l) // k * k + k
            if nxt < n: D[nxt] = D[nxt] * pow(v, MOD - 2, MOD) % MOD
        for i in range(1, n): D[i] = D[i] * D[i - 1] % MOD                             # SAI: bước 1
        for i in range(n): mul[i] = mul[i] * D[i] % MOD
    return reduce(lambda x, y: x ^ y, (a * m % MOD for a, m in zip(nums, mul)), 0)

a, qs = [1, 2, 3, 4], [[0, 3, 2, 5]]
print("sai:", wrong_stride(a, qs), "| đúng:", brute(a, qs))
assert wrong_stride(a, qs) != brute(a, qs)

sai: 20 | đúng: 12


### Lỗi 4 — Quên lấy modulo trước khi XOR

In [17]:
a, qs = [10**9, 7], [[0, 0, 1, 10**5]]
raw = (10**9 * 10**5) ^ 7; good = brute(a, qs)
print("XOR không modulo:", raw, "| đúng:", good)
assert raw != good

XOR không modulo: 100000000000007 | đúng: 999300000


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Truy vấn trên cấp số cộng với bước `k` ⇒ **chia căn theo `k`**: `k` lớn đi thẳng, `k` nhỏ gộp theo lớp đồng dư.
- Cập nhật đoạn bằng phép nhân modulo nguyên tố ⇒ mảng hiệu nhân + nghịch đảo.
- Phép toán giao hoán ⇒ xử lý offline, gom theo nhóm.
- Python: cố gắng để vòng lặp nóng chỉ có phép nhân/modulo trên list (tránh dict).

## 15. Biến thể của bài

1. **Bản I** (`n, q ≤ 10^3`): mô phỏng trực tiếp.
2. **Cộng** thay vì nhân: mảng hiệu cộng theo bước `k`, không cần nghịch đảo.
3. **Truy vấn xen kẽ** (hỏi giá trị giữa chừng): cần cấu trúc online — chia căn với lazy cho từng `k` nhỏ.
4. **Modulo không nguyên tố**: `v` có thể không khả nghịch ⇒ dùng segment tree / sparse product thay cho mảng hiệu.

In [18]:
# Biến thể 2: phép cộng theo bước k
def add_queries(nums, queries):
    n = len(nums); add = [0] * n; small = defaultdict(list)
    for l, r, k, v in queries: small[k].append((l, r, v))
    for k, lst in small.items():
        D = [0] * n
        for l, r, v in lst:
            D[l] += v; nxt = l + (r - l) // k * k + k
            if nxt < n: D[nxt] -= v
        for i in range(k, n): D[i] += D[i - k]
        for i in range(n): add[i] += D[i]
    return [a + d for a, d in zip(nums, add)]

def add_brute(nums, queries):
    a = nums[:]
    for l, r, k, v in queries:
        for i in range(l, r + 1, k): a[i] += v
    return a

rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 12); a = [rng.randint(0, 9) for _ in range(n)]; qs = []
    for _ in range(6):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r, rng.randint(1, n), rng.randint(1, 9)])
    assert add_queries(a, qs) == add_brute(a, qs)
print("Biến thể 2 OK")

Biến thể 2 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Mô phỏng | Định nghĩa | `O(n q)` | Bản I, kiểm chứng |
| 2. Chia căn dày | Mảng hiệu theo `k` | `O(n √q)` | **Lời giải chuẩn** |
| 3. Chia căn thưa | Sự kiện theo `(k, res)` | `O(n √q)` | Khi nhiều `k` nhưng ít truy vấn mỗi lớp |
| 4. Khảo sát `B` | Thực nghiệm | — | Chọn ngưỡng cho dữ liệu thật |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [1,2,3,4]`, `queries = [[0,3,3,2]]`.

**Bài 2 (tính tay).** `nums = [1,1,1,1,1]`, `queries = [[1,4,2,10],[0,4,4,3]]`.

**Bài 3 (điền TODO).** Viết `final_array(nums, queries)` trả về mảng cuối (không XOR).

**Bài 4 (tự viết lại).** Viết lại Cách 2 nhưng chọn `B` theo công thức `√(Σ (r−l)/k)` ước lượng từ dữ liệu.

**Bài 5 (mini challenge).** Tính **tổng** (không phải XOR) mảng cuối modulo `p`, với `n = 10^5` nhưng chỉ cần `O(q √n)` (gợi ý: không cần mảng cuối từng phần tử nếu bạn khéo — hoặc dùng luôn Cách 2).

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def final_array(nums, queries):
    # TODO
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def final_sum(nums, queries):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chạm chỉ số `0, 3`. (2) Mảng `[2, 2, 3, 8]`. (3) XOR `2^2^3^8 = 11`.

**Bài 2.** (1) Truy vấn 1 chạm `1, 3` (×10). (2) Truy vấn 2 chạm `0, 4` (×3). (3) `[3, 10, 1, 10, 3]` ⇒ XOR `1`.

**Bài 3.** (1) Như Cách 2 nhưng trả `[a·m mod p]`. (2) So với mô phỏng. (3) Dùng lại `mul`.

**Bài 4.** (1) `S = Σ ((r−l)//k + 1)` = số bước mô phỏng. (2) Mô phỏng nếu `S` nhỏ. (3) Nếu lớn, chọn `B` cân bằng.

**Bài 5.** (1) Tính `mul` như Cách 2. (2) `Σ nums[i]·mul[i]`. (3) Lấy modulo.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().xorAfterQueries([1, 2, 3, 4], [[0, 3, 3, 2]]) == 11
assert Solution().xorAfterQueries([1, 1, 1, 1, 1], [[1, 4, 2, 10], [0, 4, 4, 3]]) == 1 == brute([1, 1, 1, 1, 1], [[1, 4, 2, 10], [0, 4, 4, 3]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def final_array_sol(nums, queries, B=None):
    n = len(nums)
    if B is None: B = max(1, int(math.isqrt(len(queries))))
    mul = [1] * n; small = defaultdict(list)
    for l, r, k, v in queries:
        if k > B:
            for i in range(l, r + 1, k): mul[i] = mul[i] * v % MOD
        else: small[k].append((l, r, v))
    for k, lst in small.items():
        D = [1] * n
        for l, r, v in lst:
            D[l] = D[l] * v % MOD; nxt = l + (r - l) // k * k + k
            if nxt < n: D[nxt] = D[nxt] * pow(v, MOD - 2, MOD) % MOD
        for i in range(k, n): D[i] = D[i] * D[i - k] % MOD
        for i in range(n): mul[i] = mul[i] * D[i] % MOD
    return [a * m % MOD for a, m in zip(nums, mul)]

def adaptive(nums, queries):
    steps = sum((r - l) // k + 1 for l, r, k, v in queries)
    if steps <= 4 * (len(nums) + len(queries)):           # mô phỏng rẻ
        return brute(nums, queries)
    return sqrt_dense(nums, queries, max(1, int(math.isqrt(len(queries)))))

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 15); a = [rng.randint(1, 100) for _ in range(n)]; qs = []
    for _ in range(rng.randint(1, 10)):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r, rng.randint(1, n), rng.randint(1, 100)])
    b = a[:]
    for l, r, k, v in qs:
        for i in range(l, r + 1, k): b[i] = b[i] * v % MOD
    assert final_array_sol(a, qs) == b and adaptive(a, qs) == brute(a, qs)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5
def final_sum_sol(nums, queries):
    return sum(final_array_sol(nums, queries)) % MOD

print("Bài 5:", final_sum_sol([2, 3, 1, 5, 4], [[1, 4, 2, 3], [0, 2, 1, 2]]), "(= 4 + 18 + 2 + 15 + 4 = 43)")
assert final_sum_sol([2, 3, 1, 5, 4], [[1, 4, 2, 3], [0, 2, 1, 2]]) == 43

Bài 5: 43 (= 4 + 18 + 2 + 15 + 4 = 43)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
B = isqrt(q); mul = [1]*n
k > B: nhân trực tiếp tại l, l+k, ..., <= r
k <= B: D_k[l] *= v; nxt = l + (r-l)//k*k + k; if nxt < n: D_k[nxt] *= inv(v)
mỗi k nhỏ: for i in range(k, n): D[i] *= D[i-k]; mul[i] *= D[i]
ans = XOR(nums[i] * mul[i] % MOD)
```

**Checklist:**

- [ ] Phần tử cuối là `l + ⌊(r−l)/k⌋·k`?
- [ ] Nghịch đảo bằng `pow(v, MOD−2, MOD)`?
- [ ] Quét tích tiền tố theo **bước `k`**?
- [ ] Lấy modulo trước khi XOR?
- [ ] Đã so với mô phỏng với nhiều ngưỡng `B`?